# **Week 9: Flask**

**URLs Structure**:
```html
https://www.example.com/path or https://www.example.com/route/.. or https://www.example.com/file.html
https://www.example.com/route?key=value
https://www.example.com/route?key=value&key=value
```

## **FLASK**
A python framework used for build web-servers.

### **Installation**
```bash
pip install flash
```
**Command to start up a server**:
```bash
flash run
flask run --reload # hot reload
```

**Components**:
- app.py
- requirements.txt

### **Routes: `get` vs `post`**

In [ ]:
# basic flask app

from flask import Flask # type: ignore

# turn this file (i.e __name__) into a flask app
app = Flask(__name__)

@app.route("/")
def index():
    return "hello, world"

- `@app.route("/)`: a function called a decorator that affects the lines of code below it (usually the next function). Here it associates the `index()` function to `/` route.

In [ ]:
# v1
from flask import Flask, render_template, request # type: ignore

app = Flask(__name__)

@app.route("/")
def index():
    if "name" in request.args:
        name = request.args["name"]
    else:
        name = "World"
    return render_template("index.html", name=name)

- `request`: a variable that provides access to URL (http) parameters (the value passed in the URL after the question mark).
- the 2nd parameter of `render_template` pipes the "name" url (http) parameter into an HTML markup like so: `<p>Hello {{ name }} </p>`

> The reason for embedding data through URL (http) parameters is because of how the `get` method for html `form` works, it expects the necessary data to passed along in the URL by default.

In [ ]:
# v2
from flask import Flask, render_template, request   # type: ignore

app = Flask(__name__)


@app.route("/")
def index():
    name = request.args.get("name", "world")
    return render_template("index.html", name=name)

- Line9:  `request.args.get("name", "world")`: "name" is the URL parameter, "world" is the placeholder value if no name value is passed

#### **Jinja**
Flask uses `jinja` for templating in html.

- {% block body %}{% endblock %}
- {{ name }}
```html
{% extends "layout.html" %}

{% block body %}
    <i>Hello {{name}} </i>

    {% if name %}
        <b> {{ name }} </b>
    {% else %}
        world
    {% endif %}

{% endblock %}

```

**Resources**:  
https://jinja.palletsprojects.com/templates/

In [ ]:
# v3

from flask import Flask, render_template, request # type: ignore

app = Flask(__name__)


@app.route("/") # home route
def index():
    return render_template("index.html")

@app.route("/greet", methods=["POST"])
def greet():
    return render_template("greet.html", name=request.form.get("name", "world")) 

- `request.args`: accessing http parameters via `get`
- `request.form`: accessing http parameters via `post`. Using the `post`, now data submitted won't appear in the URL bar. Using `post` is crucial for data sensitive actions but using `get` for things like visiting the home routes are fine.

### **`get` and `post` support for a single route** 

In [ ]:
from flask import Flask, render_template, request # type: ignore

app = Flask(__name__)


@app.route("/", methods=["GET", "POST"])  # home route
def index():
    if request.method == "POST":
        # Form was submitted (i.e only way is through the form)
        return render_template("greet.html", name=request.form.get("name", "world"))
    else:
        # no submission, so show form/homepage
        return render_template("index.html")

## **Froshims: A simple flask app**

In [ ]:
# app.py: v1
from flask import Flask, request, render_template # type: ignore

app = Flask(__name__)
SPORTS = ["Soccer", "Basketball", "Ultimate Frisbee"]


@app.route("/")
def index():
    return render_template("index.html", sports=SPORTS) # sports is a variable passed to html file


@app.route("/register", methods=["POST"])
def register():
    if not request.form.get("name") or request.form.get("sport") not in SPORTS:
        return render_template("failure.html")
    return render_template("success.html")


In [ ]:
<!-- templates/index.html: v1 -->
{% extends "layout.html" %}

{% block body %}
<h1>Register</h1>

<form action="/register" method="post">
    <input type="text" autofocus autocomplete="off" placeholder="Name" name="name">
    <select name="sport" id="sport-select">
        <option selected value="">Sport</option>
        {% for sport in sports %}
        <option value="{{ sport }}"> {{ sport }}</option>
        {% endfor %}
    </select>
    <button type="submit">Register</button>
</form>
{% endblock %}

In [ ]:
# app.py: v2, stores user data in a global variable in memory
from flask import Flask, request, redirect, render_template # type: ignore

app = Flask(__name__)

REGISTRANTS = {}  # storage
SPORTS = ["Soccer", "Basketball", "Ultimate Frisbee"]


@app.route("/")
def index():
    return render_template(
        "index.html", sports=SPORTS
    )  # sports is a variable passed to html file


@app.route("/register", methods=["POST"])
def register():
    # validate user's name
    name = request.form.get("name")
    if not name:
        return render_template("error.html", message="Missing name")

    # validate sport
    sport = request.form.get("sport")
    if not sport in SPORTS:
        return render_template("error.html", message="Invalid Sport")

    # remember student
    REGISTRANTS[name] = sport

    # confirmed
    return redirect("/registrants")


@app.route("/registrants")
def registrants():
    return render_template("registrants.html", registrants=REGISTRANTS)

In [ ]:
# app.py: v3,  stores data in an sqlite db, support for deregister actions
from cs50 import SQL # type: ignore
from flask import Flask, request, redirect, render_template # type: ignore

app = Flask(__name__)

db = SQL("sqlite:///froshims.db")
SPORTS = ["Soccer", "Basketball", "Ultimate Frisbee"]


@app.route("/")
def index():
    return render_template(
        "index.html", sports=SPORTS
    )  # sports is a variable passed to html file


@app.route("/deregister", methods=["POST"])
def deregister():
    id = request.form.get("id")
    if id:
        db.execute("DELETE FROM registrants WHERE id = ?", id)
    return redirect("/registrants")


@app.route("/register", methods=["POST"])
def register():
    # validate user's name
    name = request.form.get("name")
    if not name:
        return render_template("error.html", message="Missing name")

    # validate sport
    sport = request.form.get("sport")
    if not sport in SPORTS:
        return render_template("error.html", message="Invalid Sport")

    # remember student
    db.execute("INSERT INTO registrants (name, sport) VALUES(?, ?)", name, sport)

    # confirmed
    return redirect("/registrants")


@app.route("/registrants")
def registrants():
    registrants = db.execute("SELECT * FROM registrants")
    return render_template("registrants.html", registrants=registrants)


> **Read more on**: **MCV (Model View Controller)** Paradigm, used in Flask and other applications

## **Sessions**  
**Session Cookies**: an http feature, that stores a key-value pair in a browser that is used to manage sessions (a unique interaction with a single user/client) with a web application

In [ ]:
HTTP/2 200
Content-Type: text/html
Set-Cookie: session=value  # new http header

In [ ]:
# logins-app: app.py: a simple flask app that supports session, login, logout
from flask import Flask, redirect, render_template, request, session # type: ignore
from flask_session import Session # type: ignore

# Configure app
app = Flask(__name__)

# Configure session
app.config["SESSION_PERMANENT"] = False
app.config["SESSION_TYPE"] = "filesystem"
Session(app)

@app.route("/")
def index():
    return render_template("index.html", name=session.get("name") )

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        session["name"] = request.form.get("name") # `post`: gotten from form
        return redirect("/")
    return render_template("login.html") # with `get``: probably through a link or address bar, show them home-page instead

@app.route("/logout")
def logout():
    session.clear()
    return redirect("/")

- with `session`, Flask creates a copy of server code for each client session

In [ ]:
<!-- logins-app: index.html -->
{% extends "layout.html" %}
{% block body %}
    {% if name %}
        You are logged in as {{name}}, <a href="/logout">Log out</a>.
    {% else %}
        You are not logged in. <a href="/login">Login in</a>.
    {% endif %}
{% endblock%}

## **APIs (Web)**
A web application or service that provides data over http. Return data format can be:
- text
- html/xml
- json